# TPC-DS Store Sales

**Dataset:** `samples.tpcds_sf1.store_sales`, `samples.tpcds_sf1.date_dim`, `samples.tpcds_sf1.item`

**Difficulty:** Medium

**Topics:** star schema joins, aggregation, window

In [0]:
from pyspark.sql import functions as F, types as T
from pyspark.sql import Window as W

store_sales = spark.read.table("samples.tpcds_sf1.store_sales")
date_dim = spark.read.table("samples.tpcds_sf1.date_dim")
item = spark.read.table("samples.tpcds_sf1.item")

## Problem 1

Join `store_sales` with `date_dim` on `ss_sold_date_sk = d_date_sk`. Compute total net paid and total transactions per year.

**Expected output columns:**
- `d_year`
- `total_net_paid`
- `total_transactions`

In [0]:
# Problem 1 - write your solution here
# Assign your result to: result_1
result_1 = (
    store_sales
    .join(date_dim, store_sales["ss_sold_date_sk"] == date_dim["d_date_sk"])
    .groupBy("d_year")
    .agg(
        F.sum("ss_net_paid").alias("total_net_paid"),
        F.count("*").alias("total_transactions")
    )
    .orderBy(F.col("total_transactions").desc())
)

result_1.show()

In [0]:
# ── Tests for Problem 1 ──────────────────────────────────────────
assert result_1 is not None, "result_1 is None - did you assign your DataFrame?"
assert hasattr(result_1, 'columns'), "result_1 must be a Spark DataFrame"
cols = [c.lower() for c in result_1.columns]
assert 'd_year' in cols, "Missing column: d_year"
assert 'total_net_paid' in cols, "Missing column: total_net_paid"
assert 'total_transactions' in cols, "Missing column: total_transactions"
assert len(cols) == 3, f"Expected exactly 3 columns, got {len(cols)}: {cols}"
cnt = result_1.count()
assert cnt > 0, f"Expected rows > 0, got {cnt}"
min_paid = result_1.agg(F.min('total_net_paid')).collect()[0][0]
assert float(min_paid) >= 0, f"Expected total_net_paid >= 0, found min={min_paid}"
print(f"Problem 1 passed ✓  ({cnt} rows)")

## Problem 2

Join `store_sales` with `item` on `ss_item_sk = i_item_sk`. Find the top 10 items by total net paid.

**Expected output columns:**
- `i_product_name`
- `i_category`
- `total_net_paid`
- `total_quantity`

In [0]:
# Problem 2 - write your solution here
# Assign your result to: result_2

result_2 = (
    store_sales
    .join(item, store_sales["ss_item_sk"] == item["i_item_sk"])
    .filter(F.col("i_product_name").isNotNull() & F.col("i_category").isNotNull())
    .groupBy("i_product_name", "i_category")
    .agg(
        F.sum("ss_net_paid").alias("total_net_paid"),
        F.count("ss_quantity").alias("total_quantity")
    )
    .orderBy(F.col("total_net_paid").desc())
    .limit(10)
)

result_2.show(truncate=False)

In [0]:
# ── Tests for Problem 2 ──────────────────────────────────────────
assert result_2 is not None, "result_2 is None - did you assign your DataFrame?"
assert hasattr(result_2, 'columns'), "result_2 must be a Spark DataFrame"
cols = [c.lower() for c in result_2.columns]
assert 'i_product_name' in cols, "Missing column: i_product_name"
assert 'i_category' in cols, "Missing column: i_category"
assert 'total_net_paid' in cols, "Missing column: total_net_paid"
assert 'total_quantity' in cols, "Missing column: total_quantity"
assert len(cols) == 4, f"Expected exactly 4 columns, got {len(cols)}: {cols}"
cnt = result_2.count()
assert cnt > 0, f"Expected rows > 0, got {cnt}"
assert cnt <= 10, f"Expected at most 10 rows (top 10), got {cnt}"
print(f"Problem 2 passed ✓  ({cnt} rows)")

## Problem 3

Compute total sales, average discount amount, and average net profit per store (`ss_store_sk`).

**Expected output columns:**
- `ss_store_sk`
- `total_sales`
- `avg_discount`
- `avg_net_profit`

In [0]:
# Problem 3 - write your solution here
# Assign your result to: result_3

result_3 = (
    store_sales
    .groupBy("ss_store_sk")
    .agg(
        F.sum("ss_net_paid").alias("total_sales"),
        F.avg("ss_ext_discount_amt").alias("avg_discount"),
        F.avg("ss_net_profit").alias("avg_net_profit")
    )
    .orderBy(F.col("total_sales").desc())
)

result_3.show()

In [0]:
# ── Tests for Problem 3 ──────────────────────────────────────────
assert result_3 is not None, "result_3 is None - did you assign your DataFrame?"
assert hasattr(result_3, 'columns'), "result_3 must be a Spark DataFrame"
cols = [c.lower() for c in result_3.columns]
assert 'ss_store_sk' in cols, "Missing column: ss_store_sk"
assert 'total_sales' in cols, "Missing column: total_sales"
assert 'avg_discount' in cols, "Missing column: avg_discount"
assert 'avg_net_profit' in cols, "Missing column: avg_net_profit"
assert len(cols) == 4, f"Expected exactly 4 columns, got {len(cols)}: {cols}"
cnt = result_3.count()
assert cnt > 0, f"Expected rows > 0, got {cnt}"
print(f"Problem 3 passed ✓  ({cnt} rows)")

## Problem 4

Find transactions where a coupon was used (`ss_coupon_amt > 0`).

**Expected output columns:**
- `ss_ticket_number`
- `ss_item_sk`
- `ss_sales_price`
- `ss_coupon_amt`
- `ss_net_paid`

In [0]:
# Problem 4 - write your solution here
# Assign your result to: result_4

result_4 = (
    store_sales
    .filter(F.col("ss_coupon_amt") > 0)
    .select(
        "ss_ticket_number", "ss_item_sk", "ss_sales_price", "ss_coupon_amt", "ss_net_paid"
    )
)

result_4.show(truncate=False)

In [0]:
# ── Tests for Problem 4 ──────────────────────────────────────────
assert result_4 is not None, "result_4 is None - did you assign your DataFrame?"
assert hasattr(result_4, 'columns'), "result_4 must be a Spark DataFrame"
cols = [c.lower() for c in result_4.columns]
assert 'ss_ticket_number' in cols, "Missing column: ss_ticket_number"
assert 'ss_item_sk' in cols, "Missing column: ss_item_sk"
assert 'ss_sales_price' in cols, "Missing column: ss_sales_price"
assert 'ss_coupon_amt' in cols, "Missing column: ss_coupon_amt"
assert 'ss_net_paid' in cols, "Missing column: ss_net_paid"
assert len(cols) == 5, f"Expected exactly 5 columns, got {len(cols)}: {cols}"
cnt = result_4.count()
assert cnt > 0, f"Expected rows > 0, got {cnt}"
min_coupon = result_4.agg(F.min('ss_coupon_amt')).collect()[0][0]
assert float(min_coupon) > 0, f"Expected ss_coupon_amt > 0, found min={min_coupon}"
print(f"Problem 4 passed ✓  ({cnt} rows)")

## Problem 5

Join all three tables (store_sales, date_dim, item). Compute revenue per item category per year.

**Expected output columns:**
- `d_year`
- `i_category`
- `total_revenue`
- `transaction_count`

In [0]:
# Problem 5 - write your solution here
# Assign your result to: result_5

result_5 = (
    store_sales
    .join(date_dim, store_sales["ss_sold_date_sk"] == date_dim["d_date_sk"])
    .join(item, store_sales["ss_item_sk"] == item["i_item_sk"])
    .groupBy("d_year", "i_category")
    .agg(
        F.sum("ss_net_paid").alias("total_revenue"),
        F.count("*").alias("transaction_count")
    )
    .orderBy(F.col("d_year").desc(), "i_category")
)

result_5.show()

In [0]:
# ── Tests for Problem 5 ──────────────────────────────────────────
assert result_5 is not None, "result_5 is None - did you assign your DataFrame?"
assert hasattr(result_5, 'columns'), "result_5 must be a Spark DataFrame"
cols = [c.lower() for c in result_5.columns]
assert 'd_year' in cols, "Missing column: d_year"
assert 'i_category' in cols, "Missing column: i_category"
assert 'total_revenue' in cols, "Missing column: total_revenue"
assert 'transaction_count' in cols, "Missing column: transaction_count"
assert len(cols) == 4, f"Expected exactly 4 columns, got {len(cols)}: {cols}"
cnt = result_5.count()
assert cnt > 0, f"Expected rows > 0, got {cnt}"
print(f"Problem 5 passed ✓  ({cnt} rows)")

## Problem 6

Using a window function, rank items by total net paid within each category. Keep only rank <= 5.

**Expected output columns:**
- `i_category`
- `i_product_name`
- `total_net_paid`
- `rank`

In [0]:
# Problem 6 - write your solution here
# Assign your result to: result_6
w = W.partitionBy("i_category").orderBy(F.col("total_net_paid").desc())
result_6 = (
    store_sales
    .join(item, store_sales["ss_item_sk"] == item["i_item_sk"])
    .filter(F.col("i_category").isNotNull() & F.col("i_product_name").isNotNull())
    .groupBy("i_category", "i_product_name")
    .agg(F.sum("ss_net_paid").alias("total_net_paid"))
    .withColumn("rank", F.rank().over(w))
    .filter(F.col("rank") <= 5)
    .orderBy("i_category", "rank")
)

result_6.show()

In [0]:
# ── Tests for Problem 6 ──────────────────────────────────────────
assert result_6 is not None, "result_6 is None - did you assign your DataFrame?"
assert hasattr(result_6, 'columns'), "result_6 must be a Spark DataFrame"
cols = [c.lower() for c in result_6.columns]
assert 'i_category' in cols, "Missing column: i_category"
assert 'i_product_name' in cols, "Missing column: i_product_name"
assert 'total_net_paid' in cols, "Missing column: total_net_paid"
assert 'rank' in cols, "Missing column: rank"
assert len(cols) == 4, f"Expected exactly 4 columns, got {len(cols)}: {cols}"
cnt = result_6.count()
assert cnt > 0, f"Expected rows > 0, got {cnt}"
max_rank = result_6.agg(F.max('rank')).collect()[0][0]
assert max_rank <= 5, f"Expected rank <= 5, found max={max_rank}"
min_rank = result_6.agg(F.min('rank')).collect()[0][0]
assert min_rank >= 1, f"Expected rank >= 1, found min={min_rank}"
print(f"Problem 6 passed ✓  ({cnt} rows)")

## Problem 7

Find items where net profit is negative (loss-making transactions). Count the number of loss transactions per category.

**Expected output columns:**
- `i_category`
- `loss_count`
- `total_loss`

In [0]:
# Problem 7 - write your solution here
# Assign your result to: result_7

result_7 = (
    store_sales
    .join(item, store_sales["ss_item_sk"] == item["i_item_sk"])
    .filter(F.col("ss_net_profit") < 0)
    .groupBy("i_category")
    .agg(F.count("*").alias("loss_count"), F.sum("ss_net_profit").alias("total_loss"))
    .orderBy("total_loss")
)

result_7.show()

In [0]:
# ── Tests for Problem 7 ──────────────────────────────────────────
assert result_7 is not None, "result_7 is None - did you assign your DataFrame?"
assert hasattr(result_7, 'columns'), "result_7 must be a Spark DataFrame"
cols = [c.lower() for c in result_7.columns]
assert 'i_category' in cols, "Missing column: i_category"
assert 'loss_count' in cols, "Missing column: loss_count"
assert 'total_loss' in cols, "Missing column: total_loss"
assert len(cols) == 3, f"Expected exactly 3 columns, got {len(cols)}: {cols}"
cnt = result_7.count()
assert cnt >= 0, f"Expected rows >= 0, got {cnt}"
if cnt > 0:
    max_loss = result_7.agg(F.max('total_loss')).collect()[0][0]
    assert float(max_loss) <= 0, f"Expected total_loss <= 0 (negative profits), got max={max_loss}"
print(f"Problem 7 passed ✓  ({cnt} rows)")